In [45]:
import pandas as pd
import re, html
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import random
import joblib
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path

In [10]:
files = {
    "ceas": "data\\raw\\CEAS_08.csv",
    "enron": "data\\raw\\Enron.csv",
    "nazario": "data\\raw\\Nazario.csv",
    "nigerian": "data\\raw\\Nigerian_Fraud.csv",
    "spamassassin": "data\\raw\\SpamAssasin.csv",
}

for name, path in files.items():
    df = pd.read_csv(path)
    print(name, df.shape, list(df.columns))
    print(df["label"].value_counts(), "\n")

ceas (39154, 7) ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
label
1    21842
0    17312
Name: count, dtype: int64 

enron (29767, 3) ['subject', 'body', 'label']
label
0    15791
1    13976
Name: count, dtype: int64 

nazario (1565, 7) ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
label
1    1565
Name: count, dtype: int64 

nigerian (3332, 7) ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
label
1    3332
Name: count, dtype: int64 

spamassassin (5809, 7) ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
label
0    4091
1    1718
Name: count, dtype: int64 



In [11]:
def standardize(df, source):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    if "text_combined" in df.columns:
        df["text"] = df["text_combined"]
    else:
        df["text"] = df["subject"].fillna("") + " " + df["body"].fillna("")
    out = df[["text", "label"]].copy()
    out["source"] = source
    return out

frames = [standardize(pd.read_csv(p), n) for n, p in files.items()]
data = pd.concat(frames, ignore_index=True)

In [12]:
def clean_text(t):
    t = html.unescape(str(t))
    t = re.sub(r"<[^>]+>", " ", t)                   # remove HTML tags
    t = re.sub(r"https?://\S+|www\.\S+", " URL ", t)  # hide exact links
    t = re.sub(r"\S+@\S+", " EMAIL ", t)              # hide addresses
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t

data["text"] = data["text"].map(clean_text)
data = data.dropna(subset=["text", "label"])
data = data[data["text"].str.len() > 20]
data = data.drop_duplicates(subset="text")           # do this BEFORE splitting
data["label"] = data["label"].astype(int)
print(data.groupby(["source", "label"]).size())

source        label
ceas          0        17236
              1        17145
enron         0        15471
              1        13941
nazario       1         1547
nigerian      1         3267
spamassassin  0         4089
              1         1693
dtype: int64


In [13]:
def make_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=5,
                                  max_df=0.5, sublinear_tf=True,
                                  max_features=200000)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])

for held_out in ["ceas", "enron", "spamassassin", "nazario", "nigerian"]:
    train = data[data.source != held_out]
    test = data[data.source == held_out]
    m = make_model().fit(train.text, train.label)
    print("Held out:", held_out)
    print(classification_report(test.label, m.predict(test.text), digits=3, zero_division=0))

Held out: ceas
              precision    recall  f1-score   support

           0      0.927     0.898     0.912     17236
           1      0.900     0.929     0.915     17145

    accuracy                          0.913     34381
   macro avg      0.914     0.913     0.913     34381
weighted avg      0.914     0.913     0.913     34381

Held out: enron
              precision    recall  f1-score   support

           0      0.852     0.751     0.798     15471
           1      0.756     0.855     0.802     13941

    accuracy                          0.800     29412
   macro avg      0.804     0.803     0.800     29412
weighted avg      0.806     0.800     0.800     29412

Held out: spamassassin
              precision    recall  f1-score   support

           0      0.977     0.926     0.951      4089
           1      0.842     0.948     0.892      1693

    accuracy                          0.933      5782
   macro avg      0.909     0.937     0.921      5782
weighted avg      0.

In [14]:
train = data[data.source != "enron"]
test = data[data.source == "enron"]
m_enron = make_model().fit(train.text, train.label)

pred = m_enron.predict(test.text)
false_alarms = test[(test.label == 0) & (pred == 1)]
print("False alarms:", len(false_alarms))
for t in false_alarms.text.sample(5, random_state=1):
    print("-", t[:300], "\n")

names = np.array(m_enron.named_steps["tfidf"].get_feature_names_out())
coef = m_enron.named_steps["clf"].coef_[0]
print("Top label 1 words:", list(names[np.argsort(coef)[-30:]]))
print()
print("Top benign words:", list(names[np.argsort(coef)[:30]]))

False alarms: 3845
- hpl nominations for march 31 , 2001 and april 1 - 2 , 2001 ( see attached file : hplno 401 . xls ) ( see attached file : hplno 331 . xls ) - hplno 401 . xls - hplno 331 . xls 

- total transfer capabilities attached are the total transfer capabilities ( ttc ' s ) for july 4 , 2001 . > the attached outage information is reliable at time of posting . the attached outage information is subject to change without notice . myrna neeley administrative assistant outage coordination  

- eol account for christopher fradas ( department of energy ) louise , just wanted to let you know that the account for c . fradas has been created and the pages have been customized for us gas and power . furthermore , i have tried to reach christopher twice but have not been able to do so . i have l 

- ponderosa pine energy partners , ltd . - gas brazos is not requesting any gas volumes through september 25 , 2001 . we are forecasting to start the plant around september 26 , 2001 . mm 

- 

In [15]:
train = data[data.source != "nazario"]
test = data[data.source == "nazario"]
m_naz = make_model().fit(train.text, train.label)

pred = m_naz.predict(test.text)
missed = test[pred == 0]
print("Missed:", len(missed), "of", len(test))
for t in missed.text.sample(5, random_state=1):
    print("-", t[:300], "\n")

Missed: 515 of 1547
- your usaa checking/savings account urgent preference update look for the attached doc and open 

- important - unusual activity has been detected אנו זקוקים לעזרתך בפתרון בעיה בחשבון ה-paypal שלך. עד שתעזור לנו לפתור בעיה זו, הגבלנו באופן זמני את הפעולות שתוכל לבצע בחשבון שלך. מה קורה? זוהתה פעילות חריגה בחשבון ה-paypal שלך בתאריך 26/07/2020 בשעה 17:19 gmt 03:00 באזור petah tiqwa m, ואנו חוששים ש 

- you have [9] incoming messages - ( email ) email e-mail sync failure. e-mail sync for your e-mailbox has failed and you have [9] pending messages on the mail queue. follow the steps below to download pending emails and continue to receive emails on your email account. preview e-mails now note:this s 

- absa notifyme technical issue [#17545441] date: thursday,september 27,2018 27 september 2018 grand opening absa bank limited estatement | ecertificate |absa group limited | dear valued absa customer, you will have an sms or email-based transactional history to help you

In [16]:
def clean_v2(t):
    t = re.sub(r"\b(re|fw|fwd)\s*:\s*", " ", t)   # reply and forward prefixes
    t = re.sub(r"\bwrote\b", " ", t)               # quoting artifact
    t = re.sub(r"\d+", " num ", t)                 # years, dates, numbers
    t = re.sub(r"\s+", " ", t).strip()
    return t

data["text2"] = data["text"].map(clean_v2)

In [17]:
def shared_vocab(train_df, min_sources=3):
    counts = Counter()
    for s, g in train_df.groupby("source"):
        cv = CountVectorizer(ngram_range=(1, 2), min_df=5).fit(g["text2"])
        counts.update(cv.vocabulary_.keys())      # +1 per source that has the term
    return sorted(w for w, c in counts.items() if c >= min_sources)

def make_model_v2(vocab):
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), vocabulary=vocab,
                                  sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ])

for held_out in ["ceas", "enron", "spamassassin", "nazario", "nigerian"]:
    train = data[data.source != held_out]
    test = data[data.source == held_out]
    vocab = shared_vocab(train)                    # built from training sources only
    m = make_model_v2(vocab).fit(train.text2, train.label)
    print("Held out:", held_out, "| vocabulary size:", len(vocab))
    print(classification_report(test.label, m.predict(test.text2), digits=3, zero_division=0))

Held out: ceas | vocabulary size: 15479
              precision    recall  f1-score   support

           0      0.895     0.912     0.903     17236
           1      0.909     0.893     0.901     17145

    accuracy                          0.902     34381
   macro avg      0.902     0.902     0.902     34381
weighted avg      0.902     0.902     0.902     34381

Held out: enron | vocabulary size: 14529
              precision    recall  f1-score   support

           0      0.866     0.847     0.857     15471
           1      0.834     0.855     0.844     13941

    accuracy                          0.851     29412
   macro avg      0.850     0.851     0.851     29412
weighted avg      0.851     0.851     0.851     29412

Held out: spamassassin | vocabulary size: 18637
              precision    recall  f1-score   support

           0      0.966     0.929     0.947      4089
           1      0.843     0.922     0.881      1693

    accuracy                          0.927      5782

In [18]:
df2 = pd.read_parquet("data\\raw\\meajor_cleaned_preprocessed.parquet.gzip")
print(df2.shape)
print(list(df2.columns))
print(df2.iloc[:, :3].head(3).to_string()[:1500])
for c in df2.columns:
    if df2[c].nunique() < 10:
        print(c, df2[c].value_counts().to_dict())

(108685, 20)
['sender', 'sender_domain', 'receiver', 'receiver_domain', 'date', 'subject', 'content_types', 'body', 'urls', 'url_count', 'url_length_max', 'url_length_avg', 'url_subdom_max', 'url_subdom_avg', 'attachment_count', 'has_attachments', 'attachment_types', 'language', 'source', 'label']
                                                             sender sender_domain                                                                                                                           receiver
0  d66e9e64b006d6bca649f1c945129c42c43836872b2eadfb822b6af04b3212cb     enron.com                                                                   35c5a9fb9fba3b8737ed7cef2a87e427a73db4fca85f6be4dc6a28b031887a0d
1  0907d5c64598aa2639154ed4e1556be615669e40052a1f546d692149730bdc48     enron.com                                                                   aa2c35499eae5999bf6080453cc719a891da2bb0c3803d1110bbfc60cf68f7af
2  7c3201a5ff8c5985218f1e3f11e330dc0242bbd28c6c20930a4237cfb30

In [19]:
df2 = df2.dropna(subset=["label"]).copy()
df2["label"] = df2["label"].astype(int)

print(df2["language"].value_counts().head(10))
print()
print(df2.groupby(["source", "label"]).size())
print()
print("Missing subject:", round(df2["subject"].isna().mean(), 3),
      "| Missing body:", round(df2["body"].isna().mean(), 3))
print()
for b in df2["body"].sample(3, random_state=1):
    print(repr(str(b)[:300]), "\n")

language
en          82175
en;en       21250
ru            827
ja            602
de            383
es            332
zh            315
fr            303
tr;en         284
en;en;en      218
Name: count, dtype: int64

source  label
trec5   0        29929
        1        19654
trec6   0        11208
        1         3797
trec7   0        19513
        1        24583
dtype: int64

Missing subject: 0.013 | Missing body: 0.0

"[NAME], \n\nI would not take a [FINANCIAL_INFO] hit on the [ORGANIZATION] right now - we have just started to market these and there seems to be some interest (even if it appears to be flaky).  If we forced a [DATE] sale (drop dead date - you are fired if it doesn't happen), I think we might lose as m" 

'[[USERNAME]] all [[[USERNAME]]] [[USERNAME]],\n \n[USERNAME] [[[USERNAME]]]ou [NAME], and [NAME] [[USERNAME]] spending some [[[USERNAME]]]imes on m[[[USERNAME]]] [[USERNAME]]. Bu[[[USERNAME]]] I wan[[[USERNAME]]] [[[USERNAME]]]o move li[[[USERNAME]]][[[USERNAME]]]le

In [20]:
df2["text"] = (df2["subject"].fillna("") + " " + df2["body"].fillna("")).map(clean_text)
df2["text2"] = df2["text"].map(clean_v2)

df2 = df2[df2["text"].str.len() > 20]
df2 = df2.drop_duplicates(subset="text")

overlap = set(df2["text"]) & set(data["text"])
print("Rows after cleaning:", len(df2))
print("Exact overlap with Model 1 data:", len(overlap))

Rows after cleaning: 104238
Exact overlap with Model 1 data: 7


In [21]:
for held_out in ["trec5", "trec6", "trec7"]:
    train = df2[df2.source != held_out]
    test = df2[df2.source == held_out]
    vocab = shared_vocab(train, min_sources=2)   # only 2 training sources left each round
    m = make_model_v2(vocab).fit(train.text2, train.label)
    print("Held out:", held_out, "| vocabulary size:", len(vocab))
    print(classification_report(test.label, m.predict(test.text2), digits=3, zero_division=0))

Held out: trec5 | vocabulary size: 46780
              precision    recall  f1-score   support

           0      0.967     0.732     0.834     27320
           1      0.716     0.964     0.822     19148

    accuracy                          0.828     46468
   macro avg      0.842     0.848     0.828     46468
weighted avg      0.864     0.828     0.829     46468

Held out: trec6 | vocabulary size: 68085
              precision    recall  f1-score   support

           0      0.983     0.911     0.946     11170
           1      0.768     0.950     0.849      3452

    accuracy                          0.920     14622
   macro avg      0.876     0.931     0.898     14622
weighted avg      0.933     0.920     0.923     14622

Held out: trec7 | vocabulary size: 44100
              precision    recall  f1-score   support

           0      0.870     0.881     0.875     19324
           1      0.902     0.893     0.898     23824

    accuracy                          0.888     43148
   ma

In [22]:
raw_text = df2["subject"].fillna("") + " " + df2["body"].fillna("")
df2["n_ph"] = raw_text.str.count(r"\[+[A-Z_]+\]+")
df2["n_broken"] = raw_text.str.count(r"\[\[")

print("Average placeholders per email, by label:")
print(df2.groupby("label")["n_ph"].mean())
print()
print("Share of corrupted emails (3 or more '[['), by label:")
print((df2["n_broken"] >= 3).groupby(df2["label"]).mean())

Average placeholders per email, by label:
label
0    16.048466
1    15.108134
Name: n_ph, dtype: float64

Share of corrupted emails (3 or more '[['), by label:
label
0    0.034628
1    0.007195
Name: n_broken, dtype: float64


In [23]:
def clean_model2(subject, body):
    s = "" if pd.isna(subject) else str(subject)
    b = "" if pd.isna(body) else str(body)
    t = s + " " + b
    t = t.replace("<|URL|>", " URL ")             # keep URLs as a word
    t = re.sub(r"\[+[A-Z_]+\]+", " ", t)          # remove anonymizer placeholders
    return clean_v2(clean_text(t))

df2 = df2[df2["n_broken"] < 3].copy()             # drop corrupted rows
df2["text2"] = [clean_model2(s, b) for s, b in zip(df2["subject"], df2["body"])]
df2 = df2[df2["text2"].str.len() > 20]
df2 = df2.drop_duplicates(subset="text2")
print("Rows now:", len(df2))
print(df2.groupby(["source", "label"]).size())

Rows now: 100478
source  label
trec5   0        26454
        1        18863
trec6   0        10964
        1         3401
trec7   0        17388
        1        23408
dtype: int64


In [24]:
for held_out in ["trec5", "trec6", "trec7"]:
    train = df2[df2.source != held_out]
    test = df2[df2.source == held_out]
    vocab = shared_vocab(train, min_sources=2)
    m = make_model_v2(vocab).fit(train.text2, train.label)
    print("Held out:", held_out, "| vocabulary size:", len(vocab))
    print(classification_report(test.label, m.predict(test.text2), digits=3, zero_division=0))

Held out: trec5 | vocabulary size: 44200
              precision    recall  f1-score   support

           0      0.974     0.723     0.830     26454
           1      0.715     0.973     0.824     18863

    accuracy                          0.827     45317
   macro avg      0.844     0.848     0.827     45317
weighted avg      0.866     0.827     0.828     45317

Held out: trec6 | vocabulary size: 64420
              precision    recall  f1-score   support

           0      0.984     0.919     0.950     10964
           1      0.785     0.951     0.860      3401

    accuracy                          0.927     14365
   macro avg      0.884     0.935     0.905     14365
weighted avg      0.937     0.927     0.929     14365

Held out: trec7 | vocabulary size: 42167
              precision    recall  f1-score   support

           0      0.852     0.890     0.870     17388
           1      0.916     0.885     0.900     23408

    accuracy                          0.887     40796
   ma

In [28]:
hard = pd.read_csv("data//raw//Hard_Examples.csv")
hard["label"] = hard["label"].astype(int)
hard["text2"] = [clean_model2(s, b) for s, b in zip(hard["subject"], hard["body"])]
hard = hard[hard["text2"].str.len() > 20].drop_duplicates(subset="text2")

hard_train, hard_test = train_test_split(
    hard, test_size=0.3, stratify=hard["label"], random_state=42
)
print(hard["label"].value_counts().to_dict(), "| train:", len(hard_train), "| test:", len(hard_test))

{0: 140, 1: 50} | train: 133 | test: 57


In [26]:
vocab1 = shared_vocab(data, min_sources=3)
model1_base = make_model_v2(vocab1).fit(data.text2, data.label)

vocab2 = shared_vocab(df2, min_sources=2)
model2_base = make_model_v2(vocab2).fit(df2.text2, df2.label)

for name, mdl in [("Model 1", model1_base), ("Model 2", model2_base)]:
    print(name, "| hard test, before adding hard examples")
    print(classification_report(hard_test.label, mdl.predict(hard_test.text2),
                                digits=3, zero_division=0))

Model 1 | hard test, before adding hard examples
              precision    recall  f1-score   support

           0      1.000     0.286     0.444        42
           1      0.333     1.000     0.500        15

    accuracy                          0.474        57
   macro avg      0.667     0.643     0.472        57
weighted avg      0.825     0.474     0.459        57

Model 2 | hard test, before adding hard examples
              precision    recall  f1-score   support

           0      0.960     0.571     0.716        42
           1      0.438     0.933     0.596        15

    accuracy                          0.667        57
   macro avg      0.699     0.752     0.656        57
weighted avg      0.823     0.667     0.685        57



In [27]:
def fit_with_hard(base_df, vocab, weight=20):
    train = pd.concat([base_df[["text2", "label"]],
                       hard_train[["text2", "label"]]], ignore_index=True)
    w = np.where(np.arange(len(train)) >= len(base_df), weight, 1.0)
    m = make_model_v2(vocab)
    m.fit(train.text2, train.label, clf__sample_weight=w)
    return m

model1 = fit_with_hard(data, vocab1)
model2 = fit_with_hard(df2, vocab2)

for name, mdl in [("Model 1", model1), ("Model 2", model2)]:
    print(name, "| hard test, after adding hard examples")
    print(classification_report(hard_test.label, mdl.predict(hard_test.text2),
                                digits=3, zero_division=0))

Model 1 | hard test, after adding hard examples
              precision    recall  f1-score   support

           0      1.000     0.976     0.988        42
           1      0.938     1.000     0.968        15

    accuracy                          0.982        57
   macro avg      0.969     0.988     0.978        57
weighted avg      0.984     0.982     0.983        57

Model 2 | hard test, after adding hard examples
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        42
           1      1.000     1.000     1.000        15

    accuracy                          1.000        57
   macro avg      1.000     1.000     1.000        57
weighted avg      1.000     1.000     1.000        57



In [31]:
tv = TfidfVectorizer(ngram_range=(1, 2)).fit(hard.text2)
sim = cosine_similarity(tv.transform(hard_test.text2), tv.transform(hard_train.text2))
best = sim.max(axis=1)

print("Test emails with a close twin in training (similarity > 0.8):", (best > 0.8).sum(), "of", len(best))
print(pd.Series(best).describe().round(2))

Test emails with a close twin in training (similarity > 0.8): 32 of 57
count    57.00
mean      0.65
std       0.30
min       0.09
25%       0.30
50%       0.82
75%       0.87
max       0.92
dtype: float64


In [35]:
S = cosine_similarity(tv.transform(hard.text2))
n_groups, groups = connected_components(S > 0.6)
print("Groups:", n_groups, "for", len(hard), "emails")

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(hard, hard.label, groups))
hard_train, hard_test = hard.iloc[tr_idx], hard.iloc[te_idx]
print("train:", hard_train.label.value_counts().to_dict(),
      "| test:", hard_test.label.value_counts().to_dict())

model1 = fit_with_hard(data, vocab1)
model2 = fit_with_hard(df2, vocab2)

for name, base, mdl in [("Model 1", model1_base, model1), ("Model 2", model2_base, model2)]:
    for tag, m in [("before", base), ("after", mdl)]:
        print(name, "| grouped hard test,", tag)
        print(classification_report(hard_test.label, m.predict(hard_test.text2),
                                    digits=3, zero_division=0))

Groups: 80 for 190 emails
train: {0: 89, 1: 38} | test: {0: 51, 1: 12}
Model 1 | grouped hard test, before
              precision    recall  f1-score   support

           0      1.000     0.157     0.271        51
           1      0.218     1.000     0.358        12

    accuracy                          0.317        63
   macro avg      0.609     0.578     0.315        63
weighted avg      0.851     0.317     0.288        63

Model 1 | grouped hard test, after
              precision    recall  f1-score   support

           0      1.000     0.882     0.938        51
           1      0.667     1.000     0.800        12

    accuracy                          0.905        63
   macro avg      0.833     0.941     0.869        63
weighted avg      0.937     0.905     0.911        63

Model 2 | grouped hard test, before
              precision    recall  f1-score   support

           0      1.000     0.412     0.583        51
           1      0.286     1.000     0.444        12

    

In [36]:
fresh = pd.read_csv("data//raw//Fresh_Test.csv")
fresh["text2"] = [clean_model2(s, b) for s, b in zip(fresh["subject"], fresh["body"])]

for name, mdl in [("Model 1", model1), ("Model 2", model2)]:
    print(name, "| fresh test")
    print(classification_report(fresh["label"].astype(int), mdl.predict(fresh.text2),
                                digits=3, zero_division=0))

Model 1 | fresh test
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        12
           1      1.000     1.000     1.000        12

    accuracy                          1.000        24
   macro avg      1.000     1.000     1.000        24
weighted avg      1.000     1.000     1.000        24

Model 2 | fresh test
              precision    recall  f1-score   support

           0      0.923     1.000     0.960        12
           1      1.000     0.917     0.957        12

    accuracy                          0.958        24
   macro avg      0.962     0.958     0.958        24
weighted avg      0.962     0.958     0.958        24



In [37]:
for name, mdl in [("Model 1", model1), ("Model 2", model2)]:
    pred = mdl.predict(hard_test.text2)
    wrong = hard_test[(hard_test.label == 0) & (pred == 1)]
    print(name, "| normal emails flagged as phishing:", len(wrong))
    for s, b in zip(wrong.subject, wrong.body):
        print("-", s, "|", str(b)[:150])
    print()

Model 1 | normal emails flagged as phishing: 6
- Bank login alert | A login to your online banking account was detected. If it was you, no action is required. If it was not you, open the bank's official app or use the 
- Statement notification | Your new account statement is available. Login through your usual banking app to view it. We will never ask you to send your password or OTP by email.
- Monthly bank statement available | Your monthly statement is ready to view after you login to your banking app or official website. For your security, this email does not include a link
- New sign-in to your account | We noticed a sign-in from a new device. If this was you, no action is needed. You can review recent activity in your account settings.
- Account update: monthly bank statement available | Your monthly statement is ready to view after you login to your banking app or official website. For your security, this email does not include a link
- Monthly bank statement available notificat

In [38]:
for name, mdl, other in [("Model 1 tested on Model 2's data", model1, df2),
                         ("Model 2 tested on Model 1's data", model2, data)]:
    print(name)
    print(classification_report(other.label, mdl.predict(other.text2),
                                digits=3, zero_division=0))

Model 1 tested on Model 2's data
              precision    recall  f1-score   support

           0      0.920     0.921     0.920     54806
           1      0.905     0.904     0.904     45672

    accuracy                          0.913    100478
   macro avg      0.912     0.912     0.912    100478
weighted avg      0.913     0.913     0.913    100478

Model 2 tested on Model 1's data
              precision    recall  f1-score   support

           0      0.928     0.928     0.928     36796
           1      0.930     0.929     0.929     37593

    accuracy                          0.929     74389
   macro avg      0.929     0.929     0.929     74389
weighted avg      0.929     0.929     0.929     74389



In [39]:
for name, mdl in [("Model 1", model1), ("Model 2", model2)]:
    pred = mdl.predict(hard_test.text2)
    wrong = hard_test[(hard_test.label == 0) & (pred == 1)]
    print(name, mdl.predict_proba(wrong.text2)[:, 1].round(2))

Model 1 [0.73 0.84 0.92 0.86 0.62 0.6 ]
Model 2 [0.54 0.62 0.91 0.54 0.69 0.55 0.65]


In [40]:
tmp = data.assign(pred=model2.predict(data.text2))
for s, g in tmp.groupby("source"):
    if g.label.nunique() == 2:
        print(s, "accuracy:", round((g.label == g.pred).mean(), 3), "| n =", len(g))
    else:
        print(s, "recall:", round((g.pred == 1).mean(), 3), "| n =", len(g))

ceas accuracy: 0.911 | n = 34381
enron accuracy: 0.97 | n = 29412
nazario recall: 0.72 | n = 1547
nigerian recall: 0.988 | n = 3267
spamassassin accuracy: 0.849 | n = 5782


In [41]:
tmp2 = df2.assign(pred=model1.predict(df2.text2))
for s, g in tmp2.groupby("source"):
    print(s, "accuracy:", round((g.label == g.pred).mean(), 3), "| n =", len(g))

trec5 accuracy: 0.916 | n = 45317
trec6 accuracy: 0.933 | n = 14365
trec7 accuracy: 0.902 | n = 40796


In [42]:
def band(p):
    return "likely safe" if p <= 0.3 else "likely phishing" if p >= 0.7 else "uncertain"

for name, mdl in [("Model 1", model1), ("Model 2", model2)]:
    p = mdl.predict_proba(hard_test.text2)[:, 1]
    labels = hard_test.label.map({0: "actually safe", 1: "actually phishing"})
    print(name)
    print(pd.crosstab(labels, pd.Series(p, index=hard_test.index).map(band)))
    print()

Model 1
col_0              likely phishing  likely safe  uncertain
label                                                     
actually phishing               12            0          0
actually safe                    4           33         14

Model 2
col_0              likely phishing  likely safe  uncertain
label                                                     
actually phishing               12            0          0
actually safe                    1           42          8



In [47]:
hard_all = hard[["text2", "label"]]

def fit_final(base_df, vocab, weight=20):
    train = pd.concat([base_df[["text2", "label"]], hard_all], ignore_index=True)
    w = np.where(np.arange(len(train)) >= len(base_df), weight, 1.0)
    m = make_model_v2(vocab)
    m.fit(train.text2, train.label, clf__sample_weight=w)
    return m


final1 = fit_final(data, vocab1)    # Model A: five collections combined
final2 = fit_final(df2, vocab2)     # Model B: the parquet file

out = Path(r"D:\\Email Phishing Detector\\models")
(out / "maejor").mkdir(parents=True, exist_ok=True)
joblib.dump(final1, out / "email_classifier.joblib")
joblib.dump(final2, out / "maejor" / "maejor_tfidf_logreg.joblib")

['D:\\Email Phishing Detector\\models\\maejor\\maejor_tfidf_logreg.joblib']